<a href="https://colab.research.google.com/github/vinitkumar21032008-dot/Retail_Customer_Purchase_Analysis/blob/main/Project21_Retail_Customer_Purchase_Analysis_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 21: Online Retail Customer Purchase Analysis

**Industry:** Retail / E-Commerce
**Business scenario:** Analyze customer orders, product categories, quantities and payment details to understand buying patterns and improve sales.
**Dataset:** `21_retail_customer_orders.csv`

**How to read this notebook:** every task has a short plain-language explanation, then the code, then its output. All 15 tasks from the assignment are covered in order. Money values are in rupees (₹).

> The CSV used here is a practice dataset with the same structure as the project brief. It was built with a few missing values and repeated Order_IDs on purpose, so the cleaning steps are real. If your teacher gives you the official file, put it in the same folder with the same name and re-run everything. The code uses the column names `Order_ID, Customer_ID, Order_Date, Product_Name, Category, Quantity, Unit_Price, Payment_Method`.

In [ ]:
import numpy as np
import pandas as pd

pd.options.display.max_columns = None
pd.options.display.float_format = "{:,.2f}".format

## Task 1: Load the CSV and inspect it
First look at the data: how many rows and columns, what each column is called, and what type Python thinks each column is.

In [ ]:
df = pd.read_csv("21_retail_customer_orders.csv")

print("Shape (rows, columns):", df.shape)
print("\nColumn names:", list(df.columns))
print("\nData types:")
print(df.dtypes)
df.head()

Shape (rows, columns): (1215, 8)

Column names: ['Order_ID', 'Customer_ID', 'Order_Date', 'Product_Name', 'Category', 'Quantity', 'Unit_Price', 'Payment_Method']

Data types:
Order_ID           object
Customer_ID        object
Order_Date         object
Product_Name       object
Category           object
Quantity          float64
Unit_Price        float64
Payment_Method     object
dtype: object


,Order_ID,Customer_ID,Order_Date,Product_Name,Category,Quantity,Unit_Price,Payment_Method
0,ORD101007,CUST0289,2025-07-26,Cricket Bat,Sports,2.00,"1,899.00",UPI
1,ORD100592,CUST0109,2025-04-24,Running Shoes,Clothing,1.00,"2,999.00",UPI
2,ORD100841,CUST0160,2025-05-08,Football,Sports,2.00,699.00,UPI
3,ORD100813,CUST0360,2025-01-02,Football,Sports,1.00,699.00,Debit Card
4,ORD100448,CUST0406,2025-06-08,Running Shoes,Clothing,1.00,"2,999.00",UPI


## Task 2: Find missing values and duplicate Order_IDs
Missing values break calculations, and a repeated Order_ID means the same order may be counted twice. We check both before doing anything else.

In [ ]:
print("Missing values per column:")
print(df.isna().sum())

print("\nDuplicate Order_ID records:", df["Order_ID"].duplicated().sum())
print("Rows that are exact copies of another row:", df.duplicated().sum())

# look at a few of the repeated orders side by side
repeated = df[df["Order_ID"].duplicated(keep=False)].sort_values("Order_ID")
repeated.head(6)

Missing values per column:
Order_ID           0
Customer_ID        0
Order_Date         0
Product_Name       0
Category           0
Quantity          37
Unit_Price        36
Payment_Method     0
dtype: int64

Duplicate Order_ID records: 15
Rows that are exact copies of another row: 15


,Order_ID,Customer_ID,Order_Date,Product_Name,Category,Quantity,Unit_Price,Payment_Method
62,ORD100269,CUST0240,2025-09-06,Yoga Mat,Sports,2.00,799.00,Credit Card
776,ORD100269,CUST0240,2025-09-06,Yoga Mat,Sports,2.00,799.00,Credit Card
1067,ORD100290,CUST0005,2025-10-06,Denim Jeans,Clothing,2.00,"1,599.00",UPI
891,ORD100290,CUST0005,2025-10-06,Denim Jeans,Clothing,2.00,"1,599.00",UPI
28,ORD100343,CUST0215,2025-02-23,Python Programming Book,Books,1.00,599.00,UPI
464,ORD100343,CUST0215,2025-02-23,Python Programming Book,Books,1.00,599.00,UPI


## Task 3: Handle missing Quantity and Unit_Price
**Strategy (and why):**
- **Duplicates:** every repeated Order_ID is an exact copy of another row, so it is the same order entered twice. We keep the first copy and drop the rest.
- **Unit_Price:** each product has one fixed price in this data, so a missing price is filled with that product's own price. This is more accurate than using an overall average.
- **Quantity:** most orders are for 1 or 2 units and a few are very large, so the average would be pulled upward. The **median** is a safer "typical" value, so we use it.

In [ ]:
# 1) remove repeated orders
before = len(df)
df = df.drop_duplicates(subset="Order_ID", keep="first").reset_index(drop=True)
print(f"Rows before: {before}, after removing duplicate Order_IDs: {len(df)}")

# 2) check the assumption "one price per product", then fill Unit_Price
print("Max number of different prices for any product:", df.groupby("Product_Name")["Unit_Price"].nunique().max())
price_lookup = df.groupby("Product_Name")["Unit_Price"].agg(lambda s: s.dropna().mode().iloc[0])
df["Unit_Price"] = df["Unit_Price"].fillna(df["Product_Name"].map(price_lookup))

# 3) fill Quantity with the median
median_qty = df["Quantity"].median()
df["Quantity"] = df["Quantity"].fillna(median_qty).astype(int)
print("Median quantity used for filling:", median_qty)

print("\nMissing values after cleaning:")
print(df.isna().sum())

Rows before: 1215, after removing duplicate Order_IDs: 1200
Max number of different prices for any product: 1
Median quantity used for filling: 2.0

Missing values after cleaning:
Order_ID          0
Customer_ID       0
Order_Date        0
Product_Name      0
Category          0
Quantity          0
Unit_Price        0
Payment_Method    0
dtype: int64


## Task 4: Convert Order_Date to datetime and extract year, month and weekday
Once a column is a real date, pandas can pull out its parts with `.dt`.

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Weekday"] = df["Order_Date"].dt.day_name()

print("Date range:", df["Order_Date"].min().date(), "to", df["Order_Date"].max().date())
df[["Order_ID", "Order_Date", "Year", "Month", "Month_Name", "Weekday"]].head()

Date range: 2025-01-02 to 2025-12-31


,Order_ID,Order_Date,Year,Month,Month_Name,Weekday
0,ORD101007,2025-07-26,2025,7,July,Saturday
1,ORD100592,2025-04-24,2025,4,April,Thursday
2,ORD100841,2025-05-08,2025,5,May,Thursday
3,ORD100813,2025-01-02,2025,1,January,Thursday
4,ORD100448,2025-06-08,2025,6,June,Sunday


## Task 5: Total orders, total units sold and total revenue
Revenue for one row is `Quantity × Unit_Price`; adding it up over all rows gives total sales.

In [ ]:
total_orders = df["Order_ID"].nunique()
total_units = df["Quantity"].sum()
total_revenue = (df["Quantity"] * df["Unit_Price"]).sum()

print(f"Total orders       : {total_orders:,}")
print(f"Total units sold   : {total_units:,}")
print(f"Total sales revenue: ₹{total_revenue:,.2f}")

Total orders       : 1,200
Total units sold   : 2,500
Total sales revenue: ₹4,492,850.00


## Task 6: Create the row-wise Order_Value column
`Order_Value` stores the value of each individual order. Its total must match the revenue in Task 5, which is a good self-check.

In [ ]:
df["Order_Value"] = df["Quantity"] * df["Unit_Price"]

print("Order_Value total matches Task 5 revenue:", np.isclose(df["Order_Value"].sum(), total_revenue))
df[["Order_ID", "Product_Name", "Quantity", "Unit_Price", "Order_Value"]].head()

Order_Value total matches Task 5 revenue: True


,Order_ID,Product_Name,Quantity,Unit_Price,Order_Value
0,ORD101007,Cricket Bat,2,"1,899.00","3,798.00"
1,ORD100592,Running Shoes,1,"2,999.00","2,999.00"
2,ORD100841,Football,2,699.00,"1,398.00"
3,ORD100813,Football,1,699.00,699.00
4,ORD100448,Running Shoes,1,"2,999.00","2,999.00"


## Task 7: Top 10 products by sales revenue
Group the rows by product, add up `Order_Value`, then sort from highest to lowest.

In [ ]:
top10 = (df.groupby("Product_Name")["Order_Value"].sum()
           .sort_values(ascending=False).head(10))

top10_table = df[df["Product_Name"].isin(top10.index)].groupby("Product_Name").agg(
    Category=("Category", "first"),
    Units_Sold=("Quantity", "sum"),
    Revenue=("Order_Value", "sum"),
).sort_values("Revenue", ascending=False)
top10_table

,Category,Units_Sold,Revenue
Product_Name,,,
Running Shoes,Clothing,320,"959,680.00"
Air Fryer,Home & Kitchen,118,"648,882.00"
Cricket Bat,Sports,246,"467,154.00"
Denim Jeans,Clothing,258,"412,542.00"
Mixer Grinder,Home & Kitchen,74,"244,126.00"
Smart Watch,Electronics,38,"227,962.00"
Hair Dryer,Beauty,102,"183,498.00"
Perfume,Beauty,113,"146,787.00"
Formal Shirt,Clothing,110,"142,890.00"


## Task 8: Rank product categories by quantity sold and by revenue
A category can sell many cheap units yet earn less than a category with fewer, pricier items, so we rank on both.

In [ ]:
category_summary = df.groupby("Category").agg(
    Total_Quantity=("Quantity", "sum"),
    Total_Revenue=("Order_Value", "sum"),
)
category_summary["Quantity_Rank"] = category_summary["Total_Quantity"].rank(ascending=False, method="min").astype(int)
category_summary["Revenue_Rank"] = category_summary["Total_Revenue"].rank(ascending=False, method="min").astype(int)
category_summary.sort_values("Total_Revenue", ascending=False)

,Total_Quantity,Total_Revenue,Quantity_Rank,Revenue_Rank
Category,,,,
Clothing,721,"1,618,579.00",1,1
Home & Kitchen,321,"1,015,179.00",5,2
Sports,595,"805,405.00",2,3
Electronics,371,"598,429.00",3,4
Beauty,342,"383,108.00",4,5
Books,150,"72,150.00",6,6


## Task 9: Average order value by payment method
Average order value (AOV) is the typical amount spent per order. Comparing it across payment methods shows which customers spend more.

In [ ]:
payment_summary = (df.groupby("Payment_Method")["Order_Value"]
                     .agg(Orders="count", Avg_Order_Value="mean")
                     .sort_values("Avg_Order_Value", ascending=False))
payment_summary

,Orders,Avg_Order_Value
Payment_Method,,
Debit Card,206,"4,251.79"
Net Banking,100,"3,895.89"
Credit Card,258,"3,748.78"
Cash on Delivery,186,"3,554.60"
UPI,450,"3,553.45"


## Task 10: Busiest shopping days and peak sales month
We count orders per weekday and per calendar date, and add up revenue per month to find the peak.

In [ ]:
weekday_summary = (df.groupby("Weekday")
                     .agg(Orders=("Order_ID", "count"), Revenue=("Order_Value", "sum"))
                     .sort_values("Orders", ascending=False))
print("Orders and revenue by weekday (busiest first):")
print(weekday_summary)

print("\nTop 5 busiest calendar dates:")
print(df.groupby("Order_Date")["Order_ID"].count().sort_values(ascending=False).head(5))

Orders and revenue by weekday (busiest first):
           Orders    Revenue
Weekday                     
Sunday        186 646,085.00
Wednesday     183 662,323.00
Tuesday       175 614,960.00
Saturday      166 655,500.00
Thursday      166 664,788.00
Monday        163 606,054.00
Friday        161 643,140.00

Top 5 busiest calendar dates:
Order_Date
2025-11-30    14
2025-10-27    11
2025-12-28     9
2025-11-18     9
2025-10-11     9
Name: Order_ID, dtype: int64


In [ ]:
monthly = (df.groupby(["Month", "Month_Name"])
             .agg(Orders=("Order_ID", "count"), Revenue=("Order_Value", "sum"))
             .reset_index().sort_values("Month"))
peak_month = monthly.loc[monthly["Revenue"].idxmax()]
print(f"Peak sales month: {peak_month['Month_Name']} (revenue ₹{peak_month['Revenue']:,.0f}, {int(peak_month['Orders'])} orders)")
monthly

Peak sales month: October (revenue ₹733,348, 158 orders)


,Month,Month_Name,Orders,Revenue
0,1,January,92,"382,098.00"
1,2,February,67,"259,212.00"
2,3,March,76,"272,154.00"
3,4,April,95,"290,169.00"
4,5,May,88,"299,791.00"
5,6,June,64,"232,784.00"
6,7,July,91,"375,053.00"
7,8,August,86,"276,215.00"
8,9,September,80,"300,276.00"
9,10,October,158,"733,348.00"


## Task 11: A dictionary of sales by category and a set of unique categories
A **dictionary** stores `category → total sales` pairs. A **set** keeps only unique values, so it gives each category once.

In [ ]:
category_sales = df.groupby("Category")["Order_Value"].sum().round(2).to_dict()
print("Sales by category (dictionary):")
for cat, value in category_sales.items():
    print(f"  {cat:<16} ₹{value:>12,.2f}")

unique_categories = set(df["Category"])
print("\nUnique categories (set):", unique_categories)
print("Number of unique categories:", len(unique_categories))

Sales by category (dictionary):
  Beauty           ₹  383,108.00
  Books            ₹   72,150.00
  Clothing         ₹1,618,579.00
  Electronics      ₹  598,429.00
  Home & Kitchen   ₹1,015,179.00
  Sports           ₹  805,405.00

Unique categories (set): {'Electronics', 'Beauty', 'Home & Kitchen', 'Clothing', 'Books', 'Sports'}
Number of unique categories: 6


## Task 12: A function that returns total sales for a category
A function lets us reuse the same calculation without copying code. This one ignores capital letters, so `"electronics"` and `"Electronics"` both work, and it gives a helpful message for a category that does not exist.

In [ ]:
def category_total_sales(category_name, data=df):
    """Return the total sales (Order_Value) of one category from the cleaned data."""
    match = data[data["Category"].str.lower() == category_name.strip().lower()]
    if match.empty:
        print(f"'{category_name}' not found. Available categories: {sorted(data['Category'].unique())}")
        return None
    return round(match["Order_Value"].sum(), 2)


print("Electronics :", category_total_sales("Electronics"))
print("electronics :", category_total_sales("electronics"))
print("Toys        :", category_total_sales("Toys"))

# self-check against the dictionary from Task 11
print("\nMatches dictionary:", all(category_total_sales(c) == v for c, v in category_sales.items()))

Electronics : 598429.0
electronics : 598429.0
'Toys' not found. Available categories: ['Beauty', 'Books', 'Clothing', 'Electronics', 'Home & Kitchen', 'Sports']
Toys        : None

Matches dictionary: True


## Task 13: NumPy statistics on Order_Value
- **Mean:** the average.
- **Median:** the middle value (not affected by a few huge orders).
- **Standard deviation:** how spread out the values are. We use `ddof=1` (sample standard deviation), which matches what pandas reports.

In [ ]:
order_values = df["Order_Value"].to_numpy()

print(f"Mean              : ₹{np.mean(order_values):,.2f}")
print(f"Median            : ₹{np.median(order_values):,.2f}")
print(f"Standard deviation: ₹{np.std(order_values, ddof=1):,.2f}")
if np.mean(order_values) > np.median(order_values):
    print("\nThe mean is higher than the median, so a few large orders pull the average up (right-skewed data).")
else:
    print("\nThe mean is not higher than the median, so the data is not right-skewed.")

Mean              : ₹3,744.04
Median            : ₹2,398.00
Standard deviation: ₹4,329.61

The mean is higher than the median, so a few large orders pull the average up (right-skewed data).


## Task 14: Unusually high orders (IQR method) and order-value bands
**IQR method:** `Q1` is the value below which 25% of orders fall and `Q3` the value below which 75% fall. `IQR = Q3 − Q1`. Anything above `Q3 + 1.5 × IQR` is treated as unusually high.

**Order-value bands (documented rule):**
- **Low:** order value up to Q1
- **Medium:** above Q1 up to Q3
- **High:** above Q3

In [ ]:
q1, q3 = np.percentile(order_values, [25, 75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f"Q1 = ₹{q1:,.2f}   Q3 = ₹{q3:,.2f}   IQR = ₹{iqr:,.2f}")
print(f"Lower fence = {lower_fence:,.2f} (negative, so no order can be unusually low)")
print(f"Upper fence = ₹{upper_fence:,.2f}")

outliers = df[(df["Order_Value"] > upper_fence) | (df["Order_Value"] < lower_fence)].sort_values("Order_Value", ascending=False)
print(f"\nUnusually high orders: {len(outliers)} ({len(outliers) / len(df) * 100:.1f}% of all orders)")
outliers[["Order_ID", "Order_Date", "Product_Name", "Quantity", "Unit_Price", "Order_Value"]].head(10)

Q1 = ₹1,397.50   Q3 = ₹4,796.00   IQR = ₹3,398.50
Lower fence = -3,700.25 (negative, so no order can be unusually low)
Upper fence = ₹9,893.75

Unusually high orders: 83 (6.9% of all orders)


,Order_ID,Order_Date,Product_Name,Quantity,Unit_Price,Order_Value
909,ORD100079,2025-10-10,Air Fryer,9,"5,499.00","49,491.00"
180,ORD100977,2025-07-10,Running Shoes,13,"2,999.00","38,987.00"
1139,ORD100685,2025-10-02,Running Shoes,12,"2,999.00","35,988.00"
729,ORD100048,2025-11-08,Air Fryer,5,"5,499.00","27,495.00"
389,ORD100431,2025-05-27,Air Fryer,5,"5,499.00","27,495.00"
68,ORD100423,2025-01-06,Air Fryer,5,"5,499.00","27,495.00"
555,ORD100557,2025-10-17,Cricket Bat,13,"1,899.00","24,687.00"
1084,ORD100960,2025-10-23,Smart Watch,4,"5,999.00","23,996.00"
1182,ORD101110,2025-02-15,Smart Watch,4,"5,999.00","23,996.00"
581,ORD101039,2025-11-14,Air Fryer,4,"5,499.00","21,996.00"


In [ ]:
df["Order_Band"] = pd.cut(df["Order_Value"], bins=[-np.inf, q1, q3, np.inf], labels=["Low", "Medium", "High"])

band_summary = df.groupby("Order_Band", observed=True).agg(Orders=("Order_ID", "count"), Revenue=("Order_Value", "sum"))
band_summary["Order_Share_%"] = band_summary["Orders"] / band_summary["Orders"].sum() * 100
band_summary["Revenue_Share_%"] = band_summary["Revenue"] / band_summary["Revenue"].sum() * 100
band_summary

,Orders,Revenue,Order_Share_%,Revenue_Share_%
Order_Band,,,,
Low,300,"250,174.00",25.00,5.57
Medium,609,"1,551,776.00",50.75,34.54
High,291,"2,690,900.00",24.25,59.89


In [ ]:
# save the cleaned data (with the new columns) for reuse
df.to_csv("21_retail_customer_orders_cleaned.csv", index=False)
print("Saved cleaned data:", df.shape)

Saved cleaned data: (1200, 14)


## Task 15: A simple Product class and five recommendations
A **class** is a blueprint for an object. Our `Product` stores a name and a price, and has one method that describes it. We then build `Product` objects for the five best-selling products.

In [ ]:
class Product:
    """A simple product with a name and a price."""

    def __init__(self, name, price):
        self.name = name
        self.price = price

    def describe(self):
        return f"{self.name} costs ₹{self.price:,.0f}"


top5_products = [Product(name, price_lookup[name]) for name in top10.index[:5]]
for p in top5_products:
    print(p.describe())

Running Shoes costs ₹2,999
Air Fryer costs ₹5,499
Cricket Bat costs ₹1,899
Denim Jeans costs ₹1,599
Mixer Grinder costs ₹3,299


### Five recommendations
The numbers below are calculated from the results above (not typed in by hand), so they update automatically if the data changes. They describe patterns in the data. They show association, not proof of cause, so test an idea on a small scale before rolling it out fully.

In [ ]:
# ---- numbers used by the recommendations ----
top_cat = category_summary["Total_Revenue"].idxmax()
top_cat_rev = category_summary["Total_Revenue"].max()
top_cat_share = top_cat_rev / total_revenue * 100
top_prod = top10.index[0]

low_month = monthly.loc[monthly["Revenue"].idxmin()]
busiest_day = weekday_summary["Orders"].idxmax()
quietest_day = weekday_summary["Orders"].idxmin()

best_pay = payment_summary["Avg_Order_Value"].idxmax()
worst_pay = payment_summary["Avg_Order_Value"].idxmin()

out_rev_share = outliers["Order_Value"].sum() / total_revenue * 100
out_order_share = len(outliers) / len(df) * 100

busiest_n = weekday_summary.loc[busiest_day, "Orders"]
quietest_n = weekday_summary.loc[quietest_day, "Orders"]
weekday_gap = (busiest_n - quietest_n) / quietest_n * 100

low_rev = band_summary.loc["Low", "Revenue_Share_%"]
high_orders = band_summary.loc["High", "Order_Share_%"]
high_rev = band_summary.loc["High", "Revenue_Share_%"]

recommendations = [
    f"Focus on {top_cat}. It earns ₹{top_cat_rev:,.0f} ({top_cat_share:.1f}% of all revenue) and {top_prod} is the best-selling "
    f"product by revenue. Keep these always in stock, feature them on the home page, and bundle related items with them.",

    f"Plan around timing. {peak_month['Month_Name']} is the peak month (₹{peak_month['Revenue']:,.0f}) and {low_month['Month_Name']} is the "
    f"weakest (₹{low_month['Revenue']:,.0f}). Build stock and ad budget before the peak, and use offers to lift the slow month. "
    f"By weekday, {busiest_day} has the most orders ({busiest_n}) and {quietest_day} the fewest ({quietest_n}), but the gap is only "
    f"{weekday_gap:.0f}%, so month matters far more than weekday when planning.",

    f"Use payment-based offers. Average order value is highest for {best_pay} (₹{payment_summary.loc[best_pay, 'Avg_Order_Value']:,.0f}) "
    f"and lowest for {worst_pay} (₹{payment_summary.loc[worst_pay, 'Avg_Order_Value']:,.0f}). Consider small cashback or bundle offers "
    f"for {worst_pay} customers to raise basket size. Averages are affected by a few very large orders, so test the idea on a small "
    f"group before expanding it.",

    f"Treat large orders separately. {len(outliers)} unusually large orders (above ₹{upper_fence:,.0f}) bring {out_rev_share:.1f}% of revenue "
    f"from just {out_order_share:.1f}% of orders. First confirm they are genuine (not entry errors or fraud), then offer bulk discounts "
    f"or dedicated support to repeat bulk buyers.",

    f"Lift small orders. The lowest-value quarter of orders (up to ₹{q1:,.0f}) adds only {low_rev:.1f}% of revenue, while the High band "
    f"({high_orders:.0f}% of orders) brings {high_rev:.1f}%. A free-delivery threshold or 'buy more, save more' bundles can nudge "
    f"small-basket customers to add one more item.",
]

for i, text in enumerate(recommendations, start=1):
    print(f"{i}. {text}\n")

1. Focus on Clothing. It earns ₹1,618,579 (36.0% of all revenue) and Running Shoes is the best-selling product by revenue. Keep these always in stock, feature them on the home page, and bundle related items with them.

2. Plan around timing. October is the peak month (₹733,348) and June is the weakest (₹232,784). Build stock and ad budget before the peak, and use offers to lift the slow month. By weekday, Sunday has the most orders (186) and Friday the fewest (161), but the gap is only 16%, so month matters far more than weekday when planning.

3. Use payment-based offers. Average order value is highest for Debit Card (₹4,252) and lowest for UPI (₹3,553). Consider small cashback or bundle offers for UPI customers to raise basket size. Averages are affected by a few very large orders, so test the idea on a small group before expanding it.

4. Treat large orders separately. 83 unusually large orders (above ₹9,894) bring 29.9% of revenue from just 6.9% of orders. First confirm they are ge

## Summary of what was produced
| Deliverable (from the brief) | Where |
|---|---|
| Top-selling products table | Task 7 |
| Category-wise sales summary | Task 8 and Task 11 |
| Monthly sales trend | Task 10 (monthly table) |
| Order-value statistics | Task 13 |
| Outlier list | Task 14 |
| Five sales recommendations | Task 15 |
| Cleaned dataset | `21_retail_customer_orders_cleaned.csv` |